# 06. 冪等な書き込み - 2回実行しても壊れない

`05` の最後に、こう書きました。

> 「とりあえず1件に絞る」とだけ決めて適当に選ぶと、実行するたびに結果が変わる不安定な処理になります。

この「何度実行しても同じ結果になる」性質を **冪等性 (idempotency)** と呼びます。

なぜ気にするかというと、**バッチ処理は思っているより頻繁に2回走る** からです。

- ジョブが途中で落ちて、リトライされた
- 書き込みは成功していたのに、その後の後処理でタイムアウトし、失敗扱いで再実行された
- 「昨日の分がおかしいので流し直して」と手で再実行した
- 過去分をまとめて作り直した (バックフィル)

「1回だけ実行される」という前提は、現実には成り立ちません。
2回走っても壊れないように作る、というのがここでの話です。

このノートブックで確かめること:

1. 素直に `append` すると何が起きるか
2. `txnAppId` / `txnVersion` で、書き込みそのものを1回に制限する
3. 通し番号を戻すとどうなるか
4. ストリーミングの `foreachBatch` で、チェックポイントを消しても重複しないようにする

**前提**: `00_setup` を実行済みであること。`01` `04` `05` を読んでいること。


## 準備


In [17]:
from datetime import date

from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [18]:
CATALOG = "tech_survey"

# 1〜3章で使うテーブル
TABLE = f"{CATALOG}.silver.idempotent_orders"

# 行を作るときに毎回書くので、列の定義をまとめておく
SCHEMA = "order_id INT, product STRING, amount INT, order_date DATE"

# 「誰による書き込みか」を表す名前。Deltaはこの名前ごとに通し番号を覚える
APP_ID = "06_daily_batch"

## 1. 素直に `append` すると

まず、冪等でない書き込みがどうなるかを見ます。

9月11日の注文3件を、日次バッチで追加する場面を考えます。
一度書き込んだ後、同じ処理がもう一度走ったら何件になるか、予想してください。


In [19]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

# DDL: 04と同じく Liquid Clustering で作る
spark.sql(f"""
    CREATE TABLE {TABLE} (
        order_id INT,
        product STRING,
        amount INT,
        order_date DATE
    )
    CLUSTER BY (order_date)
""")

display(spark.table(TABLE))

,order_id,product,amount,order_date


In [20]:
# 9月11日の注文3件。これが日次バッチで届いたデータだとする
daily = spark.createDataFrame(
    [
        (1, "laptop", 150000, date(2026, 9, 11)),
        (2, "monitor", 40000, date(2026, 9, 11)),
        (3, "keyboard", 12000, date(2026, 9, 11)),
    ],
    SCHEMA,
)

(
    daily.write.format("delta")
    .mode("append")  # append...既存の行を残したまま、後ろに足す
    .saveAsTable(TABLE)
)

# 1回目の書き込み結果を確認する
display(spark.table(TABLE).orderBy("order_id"))

,order_id,product,amount,order_date
0,1,laptop,150000,2026-09-11
1,2,monitor,40000,2026-09-11
2,3,keyboard,12000,2026-09-11


In [21]:
# まったく同じ処理をもう一度実行する。ジョブがリトライされた状況
(
    daily.write.format("delta")
    .mode("append")
    .saveAsTable(TABLE)
)

# 同じ order_id の行が二重になっていないか確認する
display(spark.table(TABLE).orderBy("order_id"))

,order_id,product,amount,order_date
0,1,laptop,150000,2026-09-11
1,1,laptop,150000,2026-09-11
2,2,monitor,40000,2026-09-11
3,2,monitor,40000,2026-09-11
4,3,keyboard,12000,2026-09-11
5,3,keyboard,12000,2026-09-11


`append` は「後ろに足す」としか言っていないので、2回言えば2回足されます。当然の動きです。

ここまでで、これを避ける方法は2つ見てきました。

| | 何を指示しているか | 2回実行すると |
|---|---|---|
| `append` | 後ろに足す | 増える |
| `replaceWhere` (04) | **この範囲**を、この中身にする | 同じ結果になる |
| `replaceUsing` (04) / `mergeInto` (05) | **このキーの行**を、この中身にする | 同じ結果になる |

下2つは **書き込む中身の側から** 冪等性を作っています。
「最終的にこうなっていてほしい」を宣言するので、何回言っても同じ状態に落ち着きます。

ただし、これが使えない場面があります。

- **キーが無い**。アクセスログやセンサーの値のように、同じ内容の行が複数あって当たり前のデータ
- **キーで突き合わせたくない**。テーブルが大きいと、毎回の照合が重い
- **追記しかしない**テーブル。過去の行は原理的に変わらないので、照合する意味がない

こういうときのために、まったく別の方向からの手段が用意されています。


## 2. `txnAppId` / `txnVersion` - 書き込みに通し番号を付ける

中身ではなく、**書き込みという操作そのもの** を1回に制限するやり方です。

書き込むときに、2つをセットで渡します。

- `txnAppId` … 誰による書き込みか (処理の名前)
- `txnVersion` … その中での通し番号

Deltaはトランザクションログに「この appId は、この version まで書いた」を記録します。
そして、**すでに記録した番号以下の書き込みが来たら、黙って捨てます。**

同じ番号で2回書いてみます。何件になるか予想してください。


In [22]:
# 1章の結果が残っているので、テーブルを作り直す
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

# DDL
spark.sql(f"""
    CREATE TABLE {TABLE} (
        order_id INT,
        product STRING,
        amount INT,
        order_date DATE
    )
    CLUSTER BY (order_date)
""")

""


In [23]:
(
    daily.write.format("delta")
    .mode("append")
    .option("txnAppId", APP_ID)  # どの処理による書き込みかを表す名前
    .option("txnVersion", 1)  # 1回目の書き込み、という通し番号
    .saveAsTable(TABLE)
)

display(spark.table(TABLE).orderBy("order_id"))

,order_id,product,amount,order_date
0,1,laptop,150000,2026-09-11
1,2,monitor,40000,2026-09-11
2,3,keyboard,12000,2026-09-11


In [24]:
# 同じ内容を、同じ番号でもう一度書く
(
    daily.write.format("delta")
    .mode("append")
    .option("txnAppId", APP_ID)
    .option("txnVersion", 1)  # 番号を変えていない
    .saveAsTable(TABLE)
)

# 行が増えたかどうかを確認する
display(spark.table(TABLE).orderBy("order_id"))

,order_id,product,amount,order_date
0,1,laptop,150000,2026-09-11
1,2,monitor,40000,2026-09-11
2,3,keyboard,12000,2026-09-11


2回目は **エラーも警告も出さずに** 捨てられます。

これは意図的な設計です。リトライの目的は「1回目が成功したか分からないので、もう一度試す」ことなので、
すでに書けていた場合は何もせずに成功として返るのが、呼び出す側にとって正しい振る舞いになります。

`05` のキー重複がエラーになったのと対照的です。
あちらは **どちらが正しいか決められない** ので止める。
こちらは **何もしないのが正しい** と分かっているので黙って通す、という違いです。

本当に書き込みが起きなかったのかは、テーブルの履歴で確認できます。


In [25]:
# 何回分の書き込みが記録されているかを確認する
display(
    spark.sql(f"DESCRIBE HISTORY {TABLE}").select("version", "timestamp", "operation")
)

,version,timestamp,operation
0,1,2026-09-13 08:12:16,WRITE
1,0,2026-09-13 08:12:13,CREATE TABLE


## 3. 番号を戻すとどうなるか

通し番号なので、次の書き込みでは増やします。まず素直に、9月12日の注文を番号 `2` で追加します。


In [26]:
# 翌日の注文2件
next_day = spark.createDataFrame(
    [
        (4, "mouse", 5000, date(2026, 9, 12)),
        (5, "headset", 20000, date(2026, 9, 12)),
    ],
    SCHEMA,
)

(
    next_day.write.format("delta")
    .mode("append")
    .option("txnAppId", APP_ID)
    .option("txnVersion", 2)  # 番号を1つ進めた
    .saveAsTable(TABLE)
)

# 9月12日の2件が増えたことを確認する
display(spark.table(TABLE).orderBy("order_id"))

,order_id,product,amount,order_date
0,1,laptop,150000,2026-09-11
1,2,monitor,40000,2026-09-11
2,3,keyboard,12000,2026-09-11
3,4,mouse,5000,2026-09-12
4,5,headset,20000,2026-09-12


では、**番号を `1` に戻して** 別のデータを書いたらどうなるでしょうか。

使うのは、まだテーブルに無い9月13日のデータです。中身はこれまでと重複していません。
実行する前に予想してください。


In [27]:
# まだテーブルに無い9月13日のデータ
third_day = spark.createDataFrame(
    [(6, "webcam", 8000, date(2026, 9, 13))],
    SCHEMA,
)

(
    third_day.write.format("delta")
    .mode("append")
    .option("txnAppId", APP_ID)
    .option("txnVersion", 1)  # わざと番号を戻す
    .saveAsTable(TABLE)
)

# 9月13日の行が入ったかどうかを確認する
display(spark.table(TABLE).orderBy("order_id"))

,order_id,product,amount,order_date
0,1,laptop,150000,2026-09-11
1,2,monitor,40000,2026-09-11
2,3,keyboard,12000,2026-09-11
3,4,mouse,5000,2026-09-12
4,5,headset,20000,2026-09-12


中身が違っても、**番号だけを見て** 捨てられます。

Deltaが覚えているのは「この appId は version 2 まで書いた」という数字だけで、
何を書いたかは見ていません。`1` は `2` 以下なので、処理済みとみなされます。

つまり `txnVersion` の付け方を間違えると、**エラーも出ないままデータが落ちます。**
ここが一番怖いところです。番号には次の性質が要ります。

- **同じ処理の再実行では、同じ番号になること**。これが冪等性の根拠になる
- **違う処理では、必ず大きい番号になること**。小さいと捨てられる

日次バッチなら「対象日を数値にしたもの」(`20260911` など) がよく使われます。
同じ日の再実行では同じ番号になり、翌日は必ず大きくなるからです。

逆に `uuid` や実行時刻のような **毎回変わる値を使ってはいけません**。
再実行で別の番号になるので、重複を防げません。
番号は処理**対象**から決め、処理**した時刻**から決めない、と覚えるとよさそうです。

`txnAppId` のほうは、**処理の単位ごとに分ける** のが基本です。
別々のジョブが同じ名前を使うと、片方の番号がもう片方の書き込みを巻き込んで捨ててしまいます。


## 4. `foreachBatch` - 書き込みを自分で書く

ここからはストリーミングの話です。まず、これまで使ってこなかった書き方を1つ入れます。

`01` `02` では、ストリームの出力先を `toTable()` で指定しました。「このテーブルに入れる」という指定です。
楽ですが、できることはそれだけに限られます。

たとえば `05` でやった `mergeInto` を、流れてくるデータに対して行いたくなったとします。
ところが **ストリーミングのDataFrameに `mergeInto` は使えません。**
ストリームは終わりのないデータなので、「全体を突き合わせて更新する」という操作が成り立たないからです。

そこで用意されているのが `foreachBatch` です。
出力先のテーブルを指定する代わりに、**自分で書いた関数を渡します。**

```python
# 第1引数にそのかたまりのDataFrame、第2引数にその通し番号が渡ってくる
def 書き込み処理(batch_df, batch_id):
    ...   # この中の batch_df は「普通のDataFrame」

(
    df.writeStream
    .foreachBatch(書き込み処理)   # 出力先の代わりに、関数を渡す
    .start()
)
```

ストリームは内部で、データを **マイクロバッチ** という小さなかたまりに区切って処理しています。
`foreachBatch` は、その1かたまりごとに関数を呼んでくれます。

呼ばれた時点で、そのかたまりは **区切られた有限のデータ** です。
なので関数の中では `mergeInto` でも複数テーブルへの書き込みでも、バッチ処理と同じことが書けます。
ストリームからバッチの世界に戻るための出口、と考えると分かりやすいと思います。

`batch_id` は、そのマイクロバッチに振られた `0` から始まる通し番号です。ここが後で効いてきます。


### 自分で書くと、保証が外れる

`toTable()` のときは、同じマイクロバッチが2回流れてきても行が二重にならないよう、
SparkとDeltaが裏で面倒を見ていました。

`foreachBatch` では書き込みを自分で書くので、この面倒も自分で見ることになります。

そして「同じマイクロバッチが2回流れてくる」は、実際に起こります。

1. マイクロバッチのデータを書き込んだ
2. **その直後に落ちた**
3. チェックポイントの更新が終わっていないので、再開すると同じバッチをもう一度処理する

`01` で見たチェックポイントは「どこまで読んだか」の記録ですが、
その更新と書き込みは別々の操作なので、間に隙間が残ります。
このため Structured Streaming の保証は **at-least-once** (最低1回) です。
「ちょうど1回」にするには、書き込み側が同じものを2回受け取っても平気である必要があります。

ここで `txnVersion` が効きます。
再実行されたマイクロバッチには **同じ `batch_id`** が渡ってくるので、
`3.` で見た「同じ処理には同じ番号」が、何もしなくても満たされるからです。


### この環境では `foreachBatch` が動かない

ここで実際に動かそうとしたところ、失敗しました。調査の記録として残しておきます。

`foreachBatch` に渡すのはPythonの関数です。
SQLやDataFrameの操作はDatabricks側のSQLエンジンの中で完結しますが、
**Pythonの関数だけは「Pythonを動かす場所」が別に必要** になります。
そのためDatabricksは、隔離されたコンテナ (サンドボックス) を起動して、その中でPythonを走らせます。

そのコンテナの起動でこうなりました。

```
[ISOLATION_STARTUP_FAILURE.SANDBOX_STARTUP] Failed to start isolated execution environment.
failed to load /databricks/python3/bin/python: exec format error
```

`exec format error` は「そのバイナリを実行できない」というOSレベルのエラーです。
Pythonインタプリタ自体が起動できていないので、こちらが書いた関数は1行も実行されていません。

同じ仕組みを使う **Python UDF** でも試したところ、まったく同じエラーになりました。

```python
spark.range(3).withColumn("x", udf(lambda i: i * 2, "long")("id")).show()
```

つまり `foreachBatch` 固有の問題ではなく、
**このワークスペースではサーバーレス上でPythonを実行できない状態** ということです。
エラーメッセージ自身も `Please contact Databricks support` と言っています。

(2026-09-13 時点。一時的な障害の可能性もあるので、後日試すと直っているかもしれません)

そこで、`foreachBatch` が内部でやっていることを **自分で書き下して** 確かめます。
`foreachBatch` の仕事は、結局この3つだけです。

1. データを小さなかたまりに区切る
2. かたまりに `0` から順に番号を振る
3. その2つを引数にして、渡された関数を呼ぶ

この3つを手で書けば、確かめたいことは確かめられます。


In [28]:
# ファイル操作をDatabricks側に対して行うための道具
from databricks.sdk.errors import NotFound
from databricks.sdk.runtime import dbutils

# ストリームの取り込み先。1〜3章とは別のテーブルにする。
STREAM_TABLE = f"{CATALOG}.bronze.idempotent_events"

LANDING = f"/Volumes/{CATALOG}/ops/landing/06_idempotent"
# ストリームを起動できないのでこの章では使わない。動く環境で試すときに使う
CHECKPOINT = f"/Volumes/{CATALOG}/ops/checkpoints/06_idempotent"

# ストリーム1本につき1つ付ける名前。1〜3章のバッチとは別にする。
STREAM_APP_ID = "06_stream"

In [29]:
import json
import random
import uuid

spark.sql(f"DROP TABLE IF EXISTS {STREAM_TABLE}")

# rm は対象が無いとエラーになるので、初回実行のために受け止める
for path in (LANDING, CHECKPOINT):
    try:
        dbutils.fs.rm(path, True)
    except NotFound:
        pass

# 取り込み元のJSONを2ファイル置く。1ファイルにつき10件
for i in (1, 2):
    rows = [
        {"event_id": str(uuid.uuid4()), "amount": random.randint(1000, 50000)}
        for _ in range(10)
    ]
    dbutils.fs.put(f"{LANDING}/events_{i}.json", "\n".join(json.dumps(r) for r in rows), True)

dbutils.fs.ls(LANDING)

[FileInfo(path='/Volumes/tech_survey/ops/landing/06_idempotent/events_1.json', name='events_1.json', size=696, modificationTime=1789287325000),
 FileInfo(path='/Volumes/tech_survey/ops/landing/06_idempotent/events_2.json', name='events_2.json', size=699, modificationTime=1789287325000)]

In [30]:
# foreachBatch に渡す関数。マイクロバッチごとに、そのバッチのDataFrameと番号が渡ってくる
def write_batch(batch_df, batch_id):
    (
        batch_df.write.format("delta")
        .mode("append")
        .option("txnAppId", STREAM_APP_ID)  # このストリームの名前
        .option("txnVersion", batch_id)  # マイクロバッチの番号をそのまま通し番号にする
        .saveAsTable(STREAM_TABLE)
    )

### 本来の書き方

動く環境では、こう書きます。ここでは実行しません。

```python
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/_schema")
    .load(LANDING)
)

query = (
    df.writeStream.foreachBatch(write_batch)   # toTable の代わりに、関数を渡す
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
```

渡している `write_batch` は、すぐ上のセルで定義したものです。
**この後の手書きでも、呼ぶ関数はこれとまったく同じ** です。
変わるのは「誰がデータを区切って、誰が番号を振るか」だけになります。


In [31]:
# foreachBatch がやっていることを手で書き下す
# 1ファイル = 1マイクロバッチ とみなして、0 から番号を振りながら write_batch を呼ぶ
for batch_id, file_name in enumerate(["events_1.json", "events_2.json"]):
    batch_df = spark.read.json(f"{LANDING}/{file_name}")
    write_batch(batch_df, batch_id)

In [32]:
# 取り込まれた件数を見る
spark.table(STREAM_TABLE).count()

20

ストリームが落ちて、同じマイクロバッチをやり直した状況を作ります。

やり直しでは **同じ番号が振り直される** ので、まったく同じループをもう一度回すことになります。

`01` でチェックポイントを消したときは、ここで行が増えました。今回はどうなるか予想してください。


In [33]:
# さきほどとまったく同じループ。番号も 0, 1 が振り直される
for batch_id, file_name in enumerate(["events_1.json", "events_2.json"]):
    batch_df = spark.read.json(f"{LANDING}/{file_name}")
    write_batch(batch_df, batch_id)

In [34]:
# やり直しで件数が増えたかどうかを見る
spark.table(STREAM_TABLE).count()

20

番号が重なった書き込みは捨てられるので、同じバッチを何度やり直しても行は増えません。
ストリームが落ちて再開しても、`01` で見たような重複は起きない、ということです。

**ただし、これは万能ではありません。**

見ているのは番号だけなので、**やり直しの前に新しいデータが増えていたら、
それも同じ番号のバッチに入って、まとめて捨てられます。**
`3.` で見たのと同じことが、ストリーミングでも起きます。

`txnVersion` は「落ちて再開したときに重複しない」ための仕組みです。
チェックポイントを消してよい理由にはならないので、そこは分けて考える必要があります。


## 5. 使い分けの整理

| 手段 | 何で冪等にするか | 向いている場面 |
|---|---|---|
| `replaceWhere` (04) | 範囲を宣言して入れ替える | 日次バッチの作り直し。範囲が明確なとき |
| `replaceUsing` (04) / `mergeInto` (05) | キーで突き合わせる | キーがあり、更新が起きるデータ |
| `txnAppId` + `txnVersion` (06) | 書き込み操作に通し番号を付ける | 追記のみ。キーが無い、または照合が重いとき |

上2つは **結果の形** を宣言するので、番号の管理が要りません。使えるならこちらのほうが安全です。
`txnVersion` は番号の付け方を間違えると静かにデータが落ちるので、
「追記しかしない」「キーで照合できない」場合の手段と考えるのがよさそうです。

なお、これらは排他ではありません。
`foreachBatch` の中で `mergeInto` しつつ冪等性も持たせる、という組み合わせも使われます。
ただし `mergeInto` は `DataFrameWriter` ではないので `.option()` を挟む場所がありません。
その場合はセッション設定 (`spark.databricks.delta.write.txnAppId` /
`spark.databricks.delta.write.txnVersion`) で渡す方法が案内されています。
ここでは試していないので、必要になったときに確かめる。


## 考えてみる

- `2.` で、2回目の書き込みがエラーにならないのは何が嬉しいのでしょうか
- 日次バッチの `txnVersion` に実行時刻 (`20260911143000` など) を使うと、何が起きるでしょうか
- `04` の `replaceWhere` が使える場面で、あえて `txnVersion` を選ぶ理由はあるでしょうか


### 答え

**Q1. エラーにならない利点**

呼び出す側が **リトライの結果を場合分けしなくてよくなる** ことです。

リトライは「1回目が成功したか分からない」から行います。
ここでエラーが返ると、呼び出す側は「本当の失敗」と「すでに書けていた」を区別して、
後者だけ無視する処理を書かなければなりません。区別を間違えれば、成功しているのに失敗扱いになります。

黙って成功が返るなら、呼び出す側は「失敗したらもう一度呼ぶ」とだけ書けばよくなります。

**Q2. 実行時刻を番号に使うと**

再実行するたびに違う番号になるので、**重複をまったく防げません。**
番号は常に大きくなるため、すべての書き込みが通ります。`append` を素で使ったのと同じ結果になります。

しかも、コードの見た目は冪等な処理になっているので、気づくのが遅れます。
番号は「同じ処理には同じ値、違う処理には大きい値」でなければなりません。

**Q3. `replaceWhere` が使えるのに `txnVersion` を選ぶか**

基本は `replaceWhere` でよいです。番号を管理しなくてよいぶん、事故が少なくなります。

`txnVersion` を選ぶ理由があるとすれば **書き込みの量** です。
`replaceWhere` は指定した範囲のファイルを書き直すので、1件足したいだけでも範囲全体が対象になります。
追記だけで済むなら、`append` + `txnVersion` のほうが書き込み量は小さくなります。

ただしこれは範囲が大きいときの話です。
まず `replaceWhere` で書いて、重くなってから考えるので十分だと思います。


## 後片付け

このノートブックで作ったものを消したいときだけ、コメントを外して実行します。


In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
# spark.sql(f"DROP TABLE IF EXISTS {STREAM_TABLE}")
# dbutils.fs.rm(LANDING, True)
# dbutils.fs.rm(CHECKPOINT, True)